# VLM-RL Analysis

Qualitative and quantitative analysis of visual reasoning improvement through RL.

Sections:
1. Load and compare evaluation results across runs
2. Side-by-side reasoning trace comparison
3. Error taxonomy visualization
4. Reward curves and training dynamics

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = Path("../results")

## 1. Benchmark Comparison

In [ ]:
# Load all summary files
summaries = {}
for f in sorted(RESULTS_DIR.glob("summary_*.json")):
    tag = f.stem.replace("summary_", "")
    with open(f) as fh:
        summaries[tag] = json.load(fh)

# Build comparison dataframe
rows = []
for tag, data in summaries.items():
    for benchmark, metrics in data.items():
        rows.append({
            "run": tag,
            "benchmark": benchmark,
            "accuracy": metrics["accuracy"],
            "correct": metrics["correct"],
            "total": metrics["total"],
        })

if rows:
    df = pd.DataFrame(rows)
    pivot = df.pivot(index="benchmark", columns="run", values="accuracy")
    display(pivot.style.format("{:.2%}").background_gradient(cmap="Greens", axis=1))
else:
    print("No results found yet. Run evaluations first.")

In [ ]:
# Bar chart comparison
if rows:
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel("Accuracy")
    ax.set_title("Benchmark Accuracy by Run")
    ax.legend(title="Run")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 2. Side-by-side Reasoning Traces

In [ ]:
def load_detailed_results(tag: str, benchmark: str = "geoqa"):
    """Load per-sample results for a specific run."""
    path = RESULTS_DIR / f"{benchmark}_detailed.json"
    if not path.exists():
        # Try tag-prefixed
        path = RESULTS_DIR / f"{tag}_{benchmark}_detailed.json"
    if not path.exists():
        path = RESULTS_DIR / f"{tag}_eval.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None


def show_trace_comparison(idx: int, runs: dict):
    """Show side-by-side traces for a given sample index."""
    for tag, results in runs.items():
        if results is None:
            continue
        samples = results.get("results", [])
        if idx < len(samples):
            s = samples[idx]
            status = "CORRECT" if s["correct"] else "WRONG"
            print(f"\n{'='*60}")
            print(f"[{tag}] [{status}] GT: {s['ground_truth']} | Pred: {s.get('predicted', 'N/A')}")
            print(f"{'='*60}")
            print(s.get("response", "N/A")[:500])


# Example usage (uncomment when results are available):
# runs = {
#     "baseline": load_detailed_results("baseline"),
#     "sft": load_detailed_results("sft"),
#     "grpo": load_detailed_results("grpo"),
# }
# show_trace_comparison(0, runs)

## 3. Error Taxonomy

In [ ]:
# Load error analysis
error_path = RESULTS_DIR / "error_analysis.json"
if error_path.exists():
    with open(error_path) as f:
        error_data = json.load(f)

    # Diagnosis pie chart
    dist = error_data.get("diagnosis_distribution", {})
    if dist:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.pie(dist.values(), labels=dist.keys(), autopct="%1.1f%%")
        ax1.set_title("Error Diagnosis Distribution")

        # Score comparison bar
        scores = error_data.get("score_summary", {})
        if scores:
            ax2.bar(scores.keys(), scores.values(), color=["#ff6b6b", "#4ecdc4", "#45b7d1"])
            ax2.set_ylim(0, 5)
            ax2.set_ylabel("Average Score (0-5)")
            ax2.set_title(f"Score Breakdown (bottleneck: {error_data.get('primary_bottleneck', '?')})")

        plt.tight_layout()
        plt.show()
else:
    print("No error analysis found. Run: python -m src.eval.analysis --results <path>")

## 4. Training Dynamics

Load WandB training logs or local logs to visualize reward curves.

In [ ]:
# If using WandB, fetch runs programmatically:
# import wandb
# api = wandb.Api()
# runs = api.runs("vlm-rl")
# for run in runs:
#     history = run.history()
#     print(f"{run.name}: {len(history)} steps")

print("Uncomment the WandB code above after training runs are complete.")
print("Or view dashboards directly at https://wandb.ai")